# BISINDO CNN+LSTM Training (Google Colab GPU)

Notebook ini menjalankan seluruh pipeline training di Colab:
1. Clone repository
2. Install dependencies
3. Aktifkan GPU (`Runtime → Change runtime type → GPU`)
4. Download dataset BISINDO alfabet
5. Upload klip gesture kata (opsional)
6. Preprocessing (MediaPipe → landmark → sliding window → `.npy`)
7. Augmentation + class balancing
8. Training CNN+LSTM dengan EarlyStopping & ReduceLROnPlateau
9. Evaluasi (classification report, confusion matrix)
10. Download `bisindo_model.h5`

## 1. Cek GPU

In [ ]:
!nvidia-smi || echo 'GPU tidak terdeteksi - aktifkan di Runtime > Change runtime type'

## 2. Clone repository & install dependencies

In [ ]:
%cd /content
!rm -rf ML_Pak_Abdi
!git clone https://github.com/buble-max/ML_Pak_Abdi.git
%cd /content/ML_Pak_Abdi
!pip install -q -r requirements.txt

## 3. Download dataset BISINDO (alfabet)

In [ ]:
!python -m dataset.download_dataset

## 4. (Opsional) Upload klip gesture kata

Rekam klip gesture kata di komputer lokal dengan `python dataset/record_word_gestures.py`, lalu kompres folder `dataset/raw_words/` menjadi `raw_words.zip` dan upload ke sini.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # pilih raw_words.zip
for name in uploaded:
    with zipfile.ZipFile(name) as zf:
        zf.extractall('dataset/')
    os.remove(name)
!ls dataset/raw_words || echo 'belum ada gesture kata'

## 5. Preprocessing → `X.npy`, `y.npy`

In [ ]:
!python -m preprocessing.landmark_extractor

## 6. Augmentation + class balancing → `X_aug.npy`, `y_aug.npy`

In [ ]:
!python -m augmentation.augment

## 7. Training CNN+LSTM

In [ ]:
import tensorflow as tf
print('TF version :', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

In [ ]:
!python -m training.train --epochs 100 --batch 64

## 8. Lihat hasil evaluasi

In [ ]:
from IPython.display import Image, display
print(open('logs/classification_report.txt').read())
display(Image('logs/confusion_matrix.png'))

## 9. Plot kurva training

In [ ]:
import json
import matplotlib.pyplot as plt

h = json.load(open('logs/history.json'))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h['loss'], label='train');      ax[0].plot(h['val_loss'], label='val')
ax[0].set_title('Loss');      ax[0].legend(); ax[0].grid(True)
ax[1].plot(h['accuracy'], label='train');  ax[1].plot(h['val_accuracy'], label='val')
ax[1].set_title('Accuracy');  ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()

## 10. Download model `.h5`

In [ ]:
from google.colab import files
files.download('model/saved/bisindo_model.h5')
files.download('model/saved/labels.json')